In [1]:
import os

In [2]:
os.environ["OPENAI_API_KEY"] = "A_TYT_NET_TOKENA("

In [3]:
from openai import OpenAI
 
client = OpenAI()
 
assistant = client.beta.assistants.create(
  name="My new assistant",
  instructions="You are my new assistant, your goal is to find answers on my questions in provided files, say that you don't know what i'm asking for if no information was found in these files.",
  model="gpt-4o",
  tools=[{"type": "file_search"}],
)

In [4]:
vector_store = client.beta.vector_stores.create(name="File store")

In [5]:
folder_path = '.\practice7_data'
file_paths = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_paths.append(os.path.join(root, file))
        
file_streams = [open(path, "rb") for path in file_paths]

file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=10, failed=0, in_progress=0, total=10)


In [6]:
assistant = client.beta.assistants.update(assistant_id=assistant.id, tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

In [23]:
# Upload the user provided file to OpenAI
 
# Create a thread and attach the file to the message
thread = client.beta.threads.create(
  messages=[
    {
      "role": "user",
      "content": "Что сделал медведь, когда сел в машину? Можешь искать не только в файлах",
    }
  ]
)
 
print(thread.tool_resources.file_search)

None


In [24]:
from typing_extensions import override
from openai import AssistantEventHandler, OpenAI
 
client = OpenAI()
 
class EventHandler(AssistantEventHandler):
    @override
    def on_text_created(self, text) -> None:
        print(f"\nassistant > ", end="", flush=True)

    @override
    def on_tool_call_created(self, tool_call):
        print(f"\nassistant > {tool_call.type}\n", flush=True)

    @override
    def on_message_done(self, message) -> None:
        message_content = message.content[0].text
        annotations = message_content.annotations
        citations = []
        for index, annotation in enumerate(annotations):
            message_content.value = message_content.value.replace(
                annotation.text, f"[{index}]"
            )
            if file_citation := getattr(annotation, "file_citation", None):
                cited_file = client.files.retrieve(file_citation.file_id)
                citations.append(f"[{index}] {cited_file.filename}")

        print(message_content.value)
        print("\n".join(citations))


with client.beta.threads.runs.stream(
    thread_id=thread.id,
    assistant_id=assistant.id,
    event_handler=EventHandler(),
) as stream:
    stream.until_done()




assistant > file_search


assistant > Информации о том, что сделал медведь, когда сел в машину, в предоставленных файлах нет.

